# FI-2010 Limit Order Book: Exploratory Data Analysis & Preprocessing
This notebook provides a comprehensive EDA of the FI-2010 High-Frequency LOB dataset, exploring class distributions, order imbalance, spread dynamics, and mid-price trajectories. Crucially, it demonstrates the structural decisions we made in our pre-processing pipeline before feeding the data into the Mixture-of-Experts (MoE) engine.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configure visual aesthetics
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Data Loading (The DecPre Structural Change)
**Major Preprocessing Action:** Instead of using the pre-normalized `Z-Score` dataset, we explicitly loaded the `NoAuction_DecPre` (Decimal Precision) subset.

**Why?** In physical markets, order volume cannot be negative. If we used Z-score data, negative volumes would have collapsed our non-linear formulas (like Order Imbalance) into infinity or flipped the mathematical signs. Loading decoupled raw data allowed us to maintain mathematical purity.

In [ ]:
# Path to Fold 1 Training Data (DecPre)
dataset_root = "data/BenchmarkDatasets"
filepath = os.path.join(dataset_root, "NoAuction", "3.NoAuction_DecPre", "NoAuction_DecPre_Training", "Train_Dst_NoAuction_DecPre_CF_1.txt")

# Load raw arrays and transpose
if os.path.exists(filepath):
    raw_data = np.loadtxt(filepath).T
    print(f"Dataset Shape: {raw_data.shape}")
    
    # Extract standard features (first 144) and labels (last 5, we care about k=10 which is index 148)
    X = raw_data[:, :144]
    y = raw_data[:, 148] - 1.0  # Align labels to 0, 1, 2
else:
    print("FI-2010 Dataset not found. Please ensure it is linked.")
    # Mock arrays for framework validation if data is missing
    X = np.random.rand(1000, 144)
    y = np.random.randint(0, 3, 1000)

## 2. Feature Engineering Reconstruction
**Major Preprocessing Action:** We manually appended 3 engineered features directly from the unnormalized arrays: `Spread`, `Imbalance_L1`, and `Imbalance_L5`.

**Why?** Basic linear algorithms (like Logistic Regression) cannot dynamically compute fractional division. By augmenting the array to 147 features with these high-conviction momentum indicators, we gave the models a non-linear target to exploit. *After this step, we then dynamically passed the entire 147-dimensional array through an `sklearn.StandardScaler` to satisfy ML convergence bounds.*

In [ ]:
ask_p1 = X[:, 0]
ask_v1 = X[:, 1]
bid_p1 = X[:, 2]
bid_v1 = X[:, 3]

# Compute Level-1 Mid-Price and Spread
mid_price = (ask_p1 + bid_p1) / 2.0
spread = ask_p1 - bid_p1

# Compute Imbalance L1
imbalance_l1 = (bid_v1 - ask_v1) / (bid_v1 + ask_v1 + 1e-8)

# Compute Imbalance L5 (Top 5 levels of depth)
ask_v_L5 = X[:, [1, 5, 9, 13, 17]].sum(axis=1)
bid_v_L5 = X[:, [3, 7, 11, 15, 19]].sum(axis=1)
imbalance_l5 = (bid_v_L5 - ask_v_L5) / (bid_v_L5 + ask_v_L5 + 1e-8)

# Create a DataFrame for easy plotting
df = pd.DataFrame({
    'MidPrice': mid_price,
    'Spread': spread,
    'Imbalance_L1': imbalance_l1,
    'Imbalance_L5': imbalance_l5,
    'Class': y
})

# Map numeric classes to labels
class_map = {0: 'Up', 1: 'Stationary', 2: 'Down'}
df['ClassLabel'] = df['Class'].map(class_map)

## 3. Class Distribution Analysis & The Macro-F1 Decision
**Major Concept:** As seen below, the target labels (`Up`, `Down`, `Stationary`) are intensely skewed. 

**Major Preprocessing Action:** This is why we configured our PyTorch Multi-Layer Perceptron to use a custom `FocalLoss` instead of Cross-Entropy, and why we evaluate our models using `Macro-F1` instead of Accuracy. Pure accuracy favors the stationary class overwhelmingly.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='ClassLabel', order=['Down', 'Stationary', 'Up'], palette='coolwarm_r')
plt.title('Distribution of Price Movement Classes (k=10)')
plt.ylabel('Number of Ticks')
plt.xlabel('Movement Direction')
plt.show()

print(df['ClassLabel'].value_counts(normalize=True))

## 4. Mid-Price Trajectory Simulation
Let's visualize the raw mid-price movement across the first 5000 milliseconds (ticks) of the day to see the microstructure volatility that our MoE Engine is trying to survive in. The green and red dots indicate where the dataset considers an actual directional trend to be occurring.

In [ ]:
start, end = 0, min(5000, len(df))
plt.figure(figsize=(14, 6))
plt.plot(df['MidPrice'].iloc[start:end], color='black', linewidth=1)

# Overlay trend colors
up_idx = df.iloc[start:end][df['ClassLabel'] == 'Up'].index
down_idx = df.iloc[start:end][df['ClassLabel'] == 'Down'].index

plt.scatter(up_idx, df['MidPrice'].iloc[up_idx], color='green', s=10, label='Predicted Up', zorder=5)
plt.scatter(down_idx, df['MidPrice'].iloc[down_idx], color='red', s=10, label='Predicted Down', zorder=5)

plt.title('Mid-Price Microstructure Trajectory (Ticks 0 to 5000)')
plt.xlabel('Tick Time')
plt.ylabel('Decimal Precision Price')
plt.legend()
plt.show()

## 5. Order Imbalance Density
Does order imbalance truly correlate with price direction? We visualize the density of L5 Imbalance separated by Class. If there's a strong separation, our explicit engineered feature was highly worth it.

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df, x='Imbalance_L5', hue='ClassLabel', fill=True, common_norm=False, palette='coolwarm')
plt.title('Distribution of Level 5 Order Imbalance by Price Movement')
plt.xlabel('Imbalance Ratio [-1 (Heavy Ask) to +1 (Heavy Bid)]')
plt.ylabel('Density')
plt.axvline(0, color='black', linestyle='--', alpha=0.5)
plt.show()

## 6. Correlation Heatmap
Checking linear correlations between our engineered Spread and Imbalances against the target classification.

In [ ]:
corr_matrix = df[['Spread', 'Imbalance_L1', 'Imbalance_L5', 'Class']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Correlation Matrix of Engineered Features')
plt.show()